In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from catboost import CatBoostClassifier, Pool

e:\Python Workspace Aur\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Data

train_df = pd.read_csv("E:/Python Workspace Aur/Playground Kaggle/playground-series-s5e11/train.csv")
test_df = pd.read_csv("E:/Python Workspace Aur/Playground Kaggle/playground-series-s5e11/test.csv")

In [3]:
TARGET = 'loan_paid_back'
ID_COL = 'id'
test_ids = test_df[ID_COL]

X = train_df.drop([ID_COL, TARGET], axis=1)
y = train_df[TARGET]
X_test = test_df.drop([ID_COL], axis=1)

In [4]:
# Handling Categorical
cat_features = list(X.select_dtypes(include=['object', 'category']).columns)
X[cat_features] = X[cat_features].fillna("Missing")
X_test[cat_features] = X_test[cat_features].fillna("Missing")

# Split Validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
print("Membuat CatBoost Pool...")
train_pool = Pool(data=X_train, label=y_train, cat_features=cat_features)
val_pool = Pool(data=X_val, label=y_val, cat_features=cat_features)
full_train_pool = Pool(data=X, label=y, cat_features=cat_features)
test_pool = Pool(data=X_test, cat_features=cat_features)

Membuat CatBoost Pool...


In [6]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS']),
        'random_strength': trial.suggest_float('random_strength', 1e-9, 10.0, log=True),
        'od_type': 'Iter',
        'od_wait': 50,
        'verbose': False,
        'eval_metric': 'AUC', # Ganti metric ke AUC agar probabilitas lebih terkalibrasi
        'allow_writing_files': False
    }
    
    if params['bootstrap_type'] == 'Bayesian':
        params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0, 10)
    elif params['bootstrap_type'] == 'Bernoulli':
        params['subsample'] = trial.suggest_float('subsample', 0.1, 1)

    model = CatBoostClassifier(**params)
    
    model.fit(
        train_pool, 
        eval_set=val_pool,
        early_stopping_rounds=50,
        verbose=False
    )
    
    # PENTING: Gunakan predict_proba untuk evaluasi
    preds_proba = model.predict_proba(val_pool)[:, 1]
    auc = roc_auc_score(y_val, preds_proba)
    return auc

print("\nMulai Optimasi Optuna...")
study = optuna.create_study(direction='maximize') # Maximize AUC
study.optimize(objective, n_trials=20) 

print(f"\nBest Params: {study.best_params}")

[I 2025-11-20 10:29:22,771] A new study created in memory with name: no-name-9719436f-7339-4ec3-83cc-d54bec6a4cdc



Mulai Optimasi Optuna...


[I 2025-11-20 10:35:37,556] Trial 0 finished with value: 0.9226054570856017 and parameters: {'iterations': 1474, 'learning_rate': 0.09808670591030309, 'depth': 5, 'l2_leaf_reg': 0.12968149128100756, 'bootstrap_type': 'MVS', 'random_strength': 5.074988896683988e-07}. Best is trial 0 with value: 0.9226054570856017.
[I 2025-11-20 10:38:31,734] Trial 1 finished with value: 0.917613677893445 and parameters: {'iterations': 574, 'learning_rate': 0.06729197684723837, 'depth': 4, 'l2_leaf_reg': 0.0014596560128495551, 'bootstrap_type': 'Bernoulli', 'random_strength': 0.3627811644446554, 'subsample': 0.23073958879025935}. Best is trial 0 with value: 0.9226054570856017.
[I 2025-11-20 10:48:29,791] Trial 2 finished with value: 0.9148666688623505 and parameters: {'iterations': 845, 'learning_rate': 0.1274681899079754, 'depth': 7, 'l2_leaf_reg': 0.005015039386333123, 'bootstrap_type': 'Bayesian', 'random_strength': 0.0039449892927018075, 'bagging_temperature': 6.054992550177003}. Best is trial 0 with


Best Params: {'iterations': 1239, 'learning_rate': 0.1965268536807473, 'depth': 5, 'l2_leaf_reg': 0.7429439681934226, 'bootstrap_type': 'MVS', 'random_strength': 1.9307666622340693e-06}


In [7]:
print("\nTraining Final Model...")
best_params = study.best_params
best_params['verbose'] = 100
best_params['allow_writing_files'] = False
best_params['eval_metric'] = 'AUC'

final_model = CatBoostClassifier(**best_params)
final_model.fit(full_train_pool)


Training Final Model...
0:	total: 296ms	remaining: 6m 7s
100:	total: 19.9s	remaining: 3m 44s
200:	total: 40.2s	remaining: 3m 27s
300:	total: 1m	remaining: 3m 8s
400:	total: 1m 20s	remaining: 2m 49s
500:	total: 1m 42s	remaining: 2m 30s
600:	total: 2m 3s	remaining: 2m 11s
700:	total: 2m 24s	remaining: 1m 51s
800:	total: 2m 45s	remaining: 1m 30s
900:	total: 3m 6s	remaining: 1m 10s
1000:	total: 3m 28s	remaining: 49.7s
1100:	total: 3m 51s	remaining: 29.1s
1200:	total: 4m 14s	remaining: 8.05s
1238:	total: 4m 23s	remaining: 0us


In [8]:
test_preds_proba = final_model.predict_proba(test_pool)[:, 1] 

submission_df = pd.DataFrame({
    'id': test_ids,
    'loan_paid_back': test_preds_proba
})

# Jangan di-astype(int) ! Biarkan float.
output_filename = "submission18_catboost_pool_optuna-part2.csv"
submission_df.to_csv(output_filename, index=False)

print(f"\nSelesai! File '{output_filename}' berisi probabilitas siap disubmit.")
print(submission_df.head())


Selesai! File 'submission18_catboost_pool_optuna-part2.csv' berisi probabilitas siap disubmit.
       id  loan_paid_back
0  593994        0.945974
1  593995        0.975908
2  593996        0.314407
3  593997        0.918954
4  593998        0.963077
